# 99 — Colab fallback (DLC SuperAnimal-Quadruped)

**Kiedy używać:** lokalna instalacja DLC/torch padła na macOS Apple Silicon. Otwórz ten plik bezpośrednio w [Google Colab](https://colab.research.google.com/) (File → Upload notebook → wybierz `99_colab_fallback.ipynb`). Free tier T4 wystarczy.

**Ten notebook NIE wymaga niczego z poc/.venv ani checkpoints/.** Wszystko pobiera świeżo z PyPI / HuggingFace.

**Cel:** wykonać punkt 1 i 2 z GATE.md (setup + sensowne keypoints) w środowisku zarządzanym przez Google.

## 1. Instalacja (~5 minut)

In [ ]:
!pip install --quiet "deeplabcut[modelzoo]>=3.0.0" "huggingface_hub>=0.24" "opencv-python-headless>=4.10"

In [ ]:
import torch, deeplabcut, sys
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__, '| CUDA dostepne:', torch.cuda.is_available())
print('DLC    :', deeplabcut.__version__)
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

## 2. Pobierz sample horse video

Opcja A: wgraj swoje video w panelu Files (lewa kolumna). Opcja B: użyj fragmentu z YouTube CC-BY przez yt-dlp.

In [ ]:
!pip install --quiet "yt-dlp>=2024.8"
# Krótki klip konia w hali jeździeckiej (CC-BY — zweryfikuj przed użyciem komercyjnym).
!yt-dlp -f 'best[ext=mp4][height<=720]' --download-sections '*0:00-0:30' -o /content/sample_horse.mp4 'https://www.youtube.com/watch?v=Dy9Tcm4S22Q'
import os
print('Sample:', '/content/sample_horse.mp4', f'({os.path.getsize("/content/sample_horse.mp4") / 1e6:.1f} MB)')

## 3. Inferencja DLC SuperAnimal-Quadruped (zero-shot)

In [ ]:
import os, deeplabcut
os.makedirs('/content/outputs', exist_ok=True)

deeplabcut.video_inference_superanimal(
    videos=['/content/sample_horse.mp4'],
    superanimal_name='superanimal_quadruped',
    model_name='hrnet_w32',
    detector_name='fasterrcnn_resnet50_fpn_v2',
    video_adapt=False,
    dest_folder='/content/outputs',
    pseudo_threshold=0.1,
)

## 4. Wzrokowa inspekcja

In [ ]:
import cv2, glob, numpy as np, matplotlib.pyplot as plt

annotated = sorted(glob.glob('/content/outputs/*labeled*.mp4'))
assert annotated, 'Nie wygenerowano annotated video — sprawdz logi cell 5.'
vid = annotated[0]
print('Annotated:', vid)

cap = cv2.VideoCapture(vid)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
idx = np.linspace(0, n - 1, 5, dtype=int)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, i in zip(axes, idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
    ok, frame = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'klatka {i}')
    ax.axis('off')
cap.release()
plt.tight_layout()
plt.savefig('/content/outputs/sample_keypoints_grid.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Pobierz wyniki na laptop

W panelu Files (lewa kolumna) prawym kliknij na `outputs/` → Download. Skopiuj annotated MP4 i grid PNG do `poc/outputs/` na laptopie i wpisz wyniki do `GATE.md`.

**Uwaga:** ten notebook nie zastępuje `01_read_my_ears_replicate.ipynb`. Punkt 3 GATE.md (Read My Ears) wymaga osobnego uruchomienia — można je zrobić w nowym Colab notebooku, ale w Fazie 0 wystarczy DLC sanity check + odznaczenie punktu 1 i 2 z GATE.